In [1]:
import time
notebook_start = time.perf_counter()

%pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

Obtaining file:///home/darshan/A6/PCSAFT_cDFT/thermoift
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for thermoift (pyproject.toml) ... done
  Created wheel for thermoift: filename=thermoift-0.2.0-0.editable-py3-none-any.whl size=1367 sha256=4e0670340bc5f7a56b48bbaf3e7031c77da118f5905ebe8fbfa8dd9335e12a3f
  Stored in directory: /tmp/pip-ephem-wheel-cache-4gcb9bkl/wheels/fd/2f/c4/54a2ee5cd16a9bf5b183bbe5c28d1b3ba4926fb0261a13e1e4
Successfully built thermoift
  Attempting uninstall: thermoift
    Found existing installation: thermoift 0.2.0
    Uninstalling thermoift-0.2.0:
      Successfully uninstalled thermoift-0.2.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from thermoift import FeedsBuilder
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("CSV_feeds")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
# ── Configuration ──
COMPONENTS  = ["CO2", "H2", "Ar", "N2", "CH4", "O2", "CO", "H2S"]
CO2_LEVELS  = (0.95, 0.96, 0.97, 0.98, 0.99)
SEED        = 58

builder = FeedsBuilder(rng_type="PCG64")

### 1. Random feeds (Dirichlet sampling)

In [4]:
df_random = builder.generate_random_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    n_random_samples=200,
    rng=SEED,
)
print(f"Random feeds: {len(df_random)}")
df_random.head()

Random feeds: 28000


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,mixture_size,active_components,feed_source,template_name
0,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
1,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
2,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
3,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
4,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet


### 2. Systematic feeds (grid at fixed step)

In [5]:
df_systematic = builder.generate_systematic_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    step=0.01,
)
print(f"Systematic feeds: {len(df_systematic)}")
df_systematic.head()

Systematic feeds: 455


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,mixture_size,active_components,feed_source,template_name
0,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
1,0.96,0.04,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
2,0.97,0.03,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
3,0.98,0.02,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
4,0.99,0.01,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid


### 3. Industrial feeds (bounds + ranges templates)

In [6]:
builder.load_industrial_templates()

df_ind_bounds = builder.industrial_feeds_from_bounds(
    components=COMPONENTS,
    co2_levels=CO2_LEVELS,
    rng=SEED,
)

df_ind_ranges = builder.industrial_feeds_from_ranges(
    components=COMPONENTS,
    rng=SEED,
)

df_industrial = pd.concat([df_ind_bounds, df_ind_ranges], ignore_index=True)
print(f"Industrial feeds: {len(df_industrial)}  (bounds={len(df_ind_bounds)}, ranges={len(df_ind_ranges)})")
df_industrial.head()

Industrial feeds: 320  (bounds=180, ranges=140)


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,feed_source,template_name,mixture_size,active_components
0,0.95,0.002273,0.007310,0.033833,0.006565,0.000001,0.000011,0.000007,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
1,0.95,0.022397,0.000164,0.001404,0.025985,0.000004,0.000019,0.000026,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
2,0.95,0.001642,0.019119,0.001904,0.027319,0.000002,0.000005,0.000009,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
3,0.95,0.020807,0.016937,0.010896,0.001232,0.000009,0.000026,0.000093,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
4,0.95,0.004155,0.018870,0.025527,0.001362,0.000009,0.000030,0.000047,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"


### 4. Combined feed (70% random / 20% systematic / 10% industrial)

In [7]:
df_all = builder.combine_feeds_random_systematic_industrial(
    components=COMPONENTS,
    n_total_target=100,
    co2_levels=CO2_LEVELS,
    rng=SEED,
)

print(f"Total feeds: {len(df_all)}")
print()
print(df_all["feed_source"].value_counts())
print()
df_all.head(10)

Total feeds: 100

feed_source
random              70
systematic          20
industrial_range     5
industrial_bound     5
Name: count, dtype: int64



,feed_source,template_name,mixture_size,active_components,CO2,H2,Ar,N2,CH4,O2,CO,H2S
0,random,dirichlet,5,"CO2,N2,CH4,CO,H2S",0.98,0.000000,0.000000,0.005977,0.004440,0.000000,0.004372,0.005211
1,random,dirichlet,4,"CO2,N2,CO,H2S",0.98,0.000000,0.000000,0.008010,0.000000,0.000000,0.003196,0.008794
2,random,dirichlet,4,"CO2,H2,CH4,CO",0.98,0.005692,0.000000,0.000000,0.006151,0.000000,0.008156,0.000000
3,random,dirichlet,5,"CO2,H2,N2,CH4,CO",0.95,0.009788,0.000000,0.018218,0.016778,0.000000,0.005215,0.000000
4,random,dirichlet,5,"CO2,H2,CH4,O2,H2S",0.96,0.020377,0.000000,0.000000,0.000013,0.018489,0.000000,0.001121
5,random,dirichlet,5,"CO2,H2,Ar,N2,O2",0.97,0.017121,0.001546,0.002449,0.000000,0.008885,0.000000,0.000000
6,random,dirichlet,4,"CO2,N2,CH4,CO",0.99,0.000000,0.000000,0.008693,0.000463,0.000000,0.000844,0.000000
7,random,dirichlet,7,"CO2,H2,Ar,N2,CH4,O2,H2S",0.96,0.007194,0.005344,0.005262,0.006272,0.014287,0.000000,0.001641
8,random,dirichlet,5,"CO2,H2,Ar,CH4,O2",0.97,0.005638,0.009757,0.000000,0.009425,0.005180,0.000000,0.000000
9,random,dirichlet,6,"CO2,H2,Ar,CH4,CO,H2S",0.99,0.000027,0.006143,0.000000,0.003329,0.000000,0.000177,0.000323


### 5. Save all feeds

In [8]:
# Save individual feeds
df_random.to_csv(OUTPUT_DIR / "Random_compositions.csv", index=False)
df_systematic.to_csv(OUTPUT_DIR / "Systematic_compositions.csv", index=False)
df_industrial.to_csv(OUTPUT_DIR / "Industrial_compositions.csv", index=False)

# Save combined feed
df_all.to_csv(OUTPUT_DIR / "Combined_compositions.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/:")
print(f"  Random_compositions.csv       : {len(df_random)} rows")
print(f"  Systematic_compositions.csv   : {len(df_systematic)} rows")
print(f"  Industrial_compositions.csv   : {len(df_industrial)} rows")
print(f"  Combined_compositions.csv     : {len(df_all)} rows")

elapsed = time.perf_counter() - notebook_start
print(f"\nNotebook runtime: {elapsed:.1f}s")

Saved to CSV_feeds/:
  Random_compositions.csv       : 28000 rows
  Systematic_compositions.csv   : 455 rows
  Industrial_compositions.csv   : 320 rows
  Combined_compositions.csv     : 100 rows

Notebook runtime: 4.1s
